# StudyBuddy Advanced — LoRA fine-tuning
This notebook trains a **small LoRA adapter** for Qwen3 1.7B with the starter StudyBuddy dataset, merges the adapter into the base weights, converts the result to GGUF, and packages it for Ollama on Windows.

**Not Astra:** this is a user-trained Qwen model, not a reproduction of proprietary Astra weights or capabilities. Training changes weights; it does not grant web access or tools by itself.

**Compute:** choose `Runtime → Change runtime type → T4 GPU` if available. Colab's free GPU access is not guaranteed and limits vary. Your AMD/low-VRAM laptop is for inference, not recommended for this training run. The starter dataset is a beginning, not enough to guarantee a large capability jump.

In [ ]:
from pathlib import Path
import torch, subprocess, os

repo = Path('/content/studybuddy-offline')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/karanbudhanawar-crypto/studybuddy-offline.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)

subprocess.run(['pip', 'install', '-q', '-r', str(repo / 'training' / 'requirements.txt')], check=True)
print('GPU available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU assigned. Select a GPU runtime and run this cell again. Free GPU availability varies.')
print('GPU:', torch.cuda.get_device_name(0))
print('Train examples:', sum(1 for line in (repo/'training/train.jsonl').open(encoding='utf-8') if line.strip()))
print('Evaluation examples:', sum(1 for line in (repo/'training/eval.jsonl').open(encoding='utf-8') if line.strip()))

## 1. Fine-tune
This uses regular LoRA on the original precision weights (not 4-bit QLoRA) to make the later GGUF merge/export path simpler. If CUDA runs out of memory, try `--max-length 768` and `--epochs 2`.

In [ ]:
OUTPUT = Path('/content/studybuddy-training-output')
cmd = [
    'python', str(repo/'training/train_lora.py'),
    '--train-file', str(repo/'training/train.jsonl'),
    '--eval-file', str(repo/'training/eval.jsonl'),
    '--output-dir', str(OUTPUT),
    '--epochs', '3',
    '--max-length', '1024',
]
print('Starting supervised fine-tuning...')
subprocess.run(cmd, check=True)
print('Adapter and merged checkpoint:', OUTPUT)

## 2. Convert to an Ollama-compatible GGUF
This builds the CPU quantizer and makes a Q4_K_M GGUF. Conversion can take several minutes and needs free disk space for the merged checkpoint and temporary GGUF files.

In [ ]:
EXPORT = OUTPUT / 'ollama-output'
cmd = [
    'python', str(repo/'training/export_to_ollama.py'),
    '--merged-model', str(OUTPUT/'merged-fp16'),
    '--output-dir', str(EXPORT),
    '--llama-cpp-dir', '/content/llama.cpp',
]
subprocess.run(cmd, check=True)
package = EXPORT / 'studybuddy-advanced-import.zip'
print('Windows import package ready:', package, 'size (GB):', round(package.stat().st_size/1e9, 2))

## 3. Download the trained model
The ZIP contains the quantized model and its Modelfile. Keep them together after extracting. This download may be over 1 GB.

In [ ]:
from google.colab import files
files.download(str(package))

## 4. Import on Windows

1. Extract `studybuddy-advanced-import.zip` into one folder.
2. Open Command Prompt in that folder.
3. Run `ollama create studybuddy-advanced-trained -f Modelfile.trained`.
4. Test with `ollama run studybuddy-advanced-trained`.
5. Open StudyBuddy, press Ctrl+Shift+R, choose **Advanced — StudyBuddy model**, and use **Connection settings → Check models**. Advanced mode prefers the trained model when it is installed.

Evaluate the result on examples that were not in the training file. If it regresses, keep the previous model and revise the dataset rather than assuming more epochs are better.